# Download ISMN in-situ soil moisture data

**MANUAL STEP:** Visit https://ismn.earth, log in, select the AOI + time range for a
pilot site, and download the data bundle (zip). Point `ISMN_data` below at that zip.
This notebook then parses it (via the vendored `utils/ISMN_funcs` reader), writes a
`stations.geojson` of station metadata, and one `sm_<station>_<instrument>_<depth>.csv`
per station/instrument/depth into the same directory.

Run once per AOI.

In [ ]:
import sys
import os
from pathlib import Path
import geopandas as gpd 
import pandas as pd
import pyproj
pyproj.datadir.set_data_dir(os.path.join(sys.prefix, "Library", "share", "proj"))

from ISMN_funcs.interface import ISMN_Interface

PROJECT_ROOT = Path.cwd()

C:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


### Download ISMN data
- Visit https://ismn.earth/en/ and log in
- Select AOI and time of interest
- Download data
- Place the resulting (.zip) file inside this repo without extracting it
- Copy-paste the filename in the cell below

In [ ]:
ISMN_data = str(PROJECT_ROOT / "Data_separate_files_header_20260529_20260831_13743_SU1V_20260901.zip") #Placeholder filename for the ISMN data file. Replace with your actual filename.

ismn_data = ISMN_Interface(ISMN_data)
ismn_out_dir = os.path.dirname(ISMN_data)

# Check the available data
print('Network: [Stations]\n', ismn_data.collection)

# Two depth bands -- SNOTEL/USCRN networks commonly have both a
# ~5cm and a ~20cm sensor, matching the 5cm-main/20cm-secondary
# depth convention used elsewhere in this pipeline.
depth_bands = [(0.04, 0.06), (0.19, 0.21)]
all_sm_ids = []
for min_depth, max_depth in depth_bands:
    all_sm_ids.extend(
        ismn_data.get_dataset_ids(variable=['soil_moisture'], min_depth=min_depth, max_depth=max_depth)
    )

sm_ids = []
for sm_id in all_sm_ids:
    insitu_ts_temp, meta_temp = ismn_data.read(sm_id, return_meta=True)
    sm_ids.append(sm_id)
        
print('Location number of the soil moisture data:', sm_ids)

Found existing ismn metadata in D:\Docs\RS_2023\NISAR4SM\python_metadata\Data_separate_files_header_20260529_20260831_13743_SU1V_20260901.csv.
Network: [Stations]
 SCAN: ['MooseInc'],
SNOTEL: ['Aniak', 'AtigunPass', 'Coldfoot', 'EagleSummit', 'GobblersKnob', 'GraniteCrk', 'ImnaviatCreek', 'LittleChenaRidge', 'McGrath', 'Nenana', 'NukaGlacier', 'PilgrimHotSprings', 'PrudhoeBay', 'RockyPoint', 'SpringCreek', 'SusitnaValleyHigh', 'TurnagainPass', 'UpperNomeCreek'],
USCRN: ['Kenai-29-ENE']


Location number of the soil moisture data: [0, 3, 5, 7, 9, 10, 13, 15, 17, 20, 23, 25, 27, 29, 31, 34, 36, 38, 40, 41, 1, 4, 6, 8, 11, 12, 14, 16, 18, 21, 24, 26, 28, 30, 32, 35, 37, 39, 43]


#### Get the metadata of each station and create a geojson file

In [ ]:
# One geometry row per *station*, not per depth -- sm_ids now has one entry
# per (station, depth) pair (see the widened depth_bands above), but site-level
# metadata (coordinates, texture, land cover, ...) doesn't depend on depth, so
# building one row per sm_id would write duplicate station rows.
seen_stations = set()
meta_stations = gpd.GeoDataFrame()
for sm_id in sm_ids:
    insitu_ts_temp, meta_temp = ismn_data.read(sm_id, return_meta=True)
    station_name = meta_temp['station'].values[0]
    if station_name in seen_stations:
        continue
    seen_stations.add(station_name)

    meta_temp_df = meta_temp.to_frame(meta_temp['station'].values[0])
    meta_temp_df.index = meta_temp_df.index.map('_'.join)
    meta_temp_df = meta_temp_df.T
    meta_temp_df['timerange_from_val'] = meta_temp_df['timerange_from_val'].values[0].strftime("%Y-%m-%dT%H:%M:%S")
    meta_temp_df['timerange_to_val'] = meta_temp_df['timerange_to_val'].values[0].strftime("%Y-%m-%dT%H:%M:%S")

    meta_temp_gdf = gpd.GeoDataFrame(meta_temp_df, geometry=gpd.points_from_xy(meta_temp_df.longitude_val, meta_temp_df.latitude_val), crs="EPSG:4326")
    meta_stations = pd.concat([meta_stations,meta_temp_gdf])

meta_stations.to_file(os.path.join(ismn_out_dir,'stations.geojson'), driver="GeoJSON")

#### Write Soil moisture to a csv file (one for each station/instrument combination)

In [4]:
for sm_id in sm_ids:
    insitu_sm_temp, meta_temp = ismn_data.read(sm_id, return_meta=True)
    station_name = meta_temp['station'].values[0]
    sm_data = insitu_sm_temp.loc[insitu_sm_temp['soil_moisture_flag'] == 'G', 'soil_moisture']
    sm_data.to_csv(os.path.join(ismn_out_dir,'sm_{}_{}_{}_{}.csv'.format(station_name,
                                                                        meta_temp.instrument.val,
                                                                        meta_temp.instrument.depth_from,
                                                                        meta_temp.instrument.depth_to)))

### Removed or modified station data after visual inspection: 

In [5]:
# temp_sm_data_df = pd.read_csv('/EoPersist/SSM_publication/NISAR_simulated_data/SDelta_23518/ISMN/sm_TwitchellAlfalfa_ThetaProbe-ML2X_0.2_0.2.csv')
# temp_sm_data_df.index = pd.to_datetime(temp_sm_data_df['date_time'])
# temp_sm_data_df.drop(columns=['date_time'], inplace = True)
# temp_sm_data_df[temp_sm_data_df<0.01]=np.nan
# temp_sm_data_df.dropna(inplace=True)
# #temp_sm_data_df = temp_sm_data_df['15-7-2012':'20-Oct-2012']
# temp_sm_data_df.plot(style='.-')